In [ ]:
import pandas as pd

bom_file = r"D:/Tushar/main_with_subs_only.xlsx"
indent_file = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"

MONTH_COL = "Feb'26 QTY"
DAYS_IN_MONTH = 28

# Read files
bom_df = pd.read_excel(bom_file)
indent_df = pd.read_excel(indent_file)

bom_df.columns = bom_df.columns.str.strip()
indent_df.columns = indent_df.columns.str.strip()

def normalize(series):
    return series.astype(str).str.strip().str.upper()

indent_df['Part number'] = normalize(indent_df['Part number'])
bom_df['Sub_Label'] = normalize(bom_df['Sub_Label'])
bom_df['Main_Label'] = normalize(bom_df['Main_Label'])

# Prepare lookup
indent_df = indent_df[['Part number', MONTH_COL]].copy()
indent_df[MONTH_COL] = pd.to_numeric(indent_df[MONTH_COL], errors='coerce').fillna(0)
indent_df['Daily'] = indent_df[MONTH_COL] / DAYS_IN_MONTH

lookup = dict(zip(indent_df['Part number'], indent_df['Daily']))

# Sequential accumulation
results = []

current_child = None
running_total = 0

for _, row in bom_df.iterrows():

    child = row['Main_Label']
    switch = row['Sub_Label']
    usage = pd.to_numeric(row['Sub_Count'], errors='coerce') or 0

    daily_switch = lookup.get(switch, 0)
    contribution = daily_switch * usage

    if current_child is None:
        current_child = child

    if child != current_child:
        results.append((current_child, running_total * 2))
        current_child = child
        running_total = 0

    running_total += contribution

# Append last child
if current_child is not None:
    results.append((current_child, running_total * 2))

result_df = pd.DataFrame(results, columns=['Child_Part', 'Two_Day_Requirement'])

print(result_df.head(20))

result_df.to_excel("Sequential_2Day_Result.xlsx", index=False)
